In [ ]:
from google.colab import files
files.upload()

! pip install kaggle
! mkdir -p ~/.kaggle
! cp kaggle.json ~/.kaggle/
! chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d birdy654/cifake-real-and-ai-generated-synthetic-images
! unzip cifake-real-and-ai-generated-synthetic-images.zip

In [ ]:
# Check dimensions of training images, since they have to be uniform
from PIL import Image
import os
train_dir = '/content/train'
trainfake_dir = '/content/train/FAKE'
trainreal_dir = '/content/train/REAL'

sizes = set()

for img in os.listdir(trainfake_dir):
  path = os.path.join(trainfake_dir, img) # Creates the full file path to an image by combining a directory path and a filename
  img = Image.open(path)
  sizes.add(img.size)

if len(sizes) == 1:
    print(f"Fake image size: {sizes.pop()}")

for img in os.listdir(trainreal_dir):
  path = os.path.join(trainreal_dir, img)
  img = Image.open(path)
  sizes.add(img.size)

if len(sizes) == 1:
    print(f"Real image size: {sizes.pop()}")


Fake image size: (32, 32)
Real image size: (32, 32)


In [ ]:
# Preprocess the data by rescaling to improve accuracy of CNN, add labels
from tensorflow.keras.preprocessing.image import ImageDataGenerator
batch_size = 32
train_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2) # Normalise pixel values, split 20% of data into validation set
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size = (32, 32),
    batch_size = batch_size,
    class_mode = 'binary', # Real vs Fake
    subset = 'training' # 80% of dataset
)

val_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size = (32, 32),
    batch_size = batch_size,
    class_mode = 'binary', # Real vs Fake
    subset = 'validation' # 80% of dataset
)

Found 80000 images belonging to 2 classes.
Found 20000 images belonging to 2 classes.


In [ ]:
print(train_generator.class_indices)

{'FAKE': 0, 'REAL': 1}


In [ ]:
# Build the CNN
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam


num_filters = 8
filter_size = 3
pooling_size = 2

model = Sequential([
  Conv2D(num_filters, filter_size, activation='relu', input_shape=(32, 32, 3)), # 32x32 RGB images
  MaxPooling2D(pool_size=pooling_size),
  Flatten(),
  Dense(1, activation='sigmoid'), # Final layer outputs 1 value (binary)
])

model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy']) # Default learning rate of 0.001

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
# Training the CNN
model.fit(
  train_generator,
  steps_per_epoch=len(train_generator),
  epochs=3,
  validation_data= val_generator,
  validation_steps=len(val_generator),
  verbose = 1,
)

Epoch 1/3
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 97s 38ms/step - accuracy: 0.7337 - loss: 0.5249 - val_accuracy: 0.8204 - val_loss: 0.4016
Epoch 2/3
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 136s 36ms/step - accuracy: 0.8352 - loss: 0.3738 - val_accuracy: 0.8472 - val_loss: 0.3563
Epoch 3/3
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 144s 37ms/step - accuracy: 0.8530 - loss: 0.3458 - val_accuracy: 0.8404 - val_loss: 0.3557


In [ ]:
model.save('/content/drive/MyDrive/CNN.keras')

In [ ]:
from google.colab import drive
from tensorflow.keras.models import load_model

drive.mount('/content/drive')
model = load_model('/content/drive/MyDrive/CNN.keras')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Load test dataset

from PIL import Image
import os
test_dir = '/content/test'
testfake_dir = '/content/test/FAKE'
testreal_dir = '/content/test/REAL'

sizes = set()

for img in os.listdir(testfake_dir):
  path = os.path.join(testfake_dir, img) # Creates the full file path to an image by combining a directory path and a filename
  img = Image.open(path)
  sizes.add(img.size)

if len(sizes) == 1:
    print(f"Fake image size: {sizes.pop()}")

for img in os.listdir(testreal_dir):
  path = os.path.join(testreal_dir, img)
  img = Image.open(path)
  sizes.add(img.size)

if len(sizes) == 1:
    print(f"Real image size: {sizes.pop()}")

from tensorflow.keras.preprocessing.image import ImageDataGenerator
test_datagen = ImageDataGenerator(rescale=1./255) # Normalise pixel values, split 20% of data into validation set
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size = (32, 32),
    batch_size = 32,
    class_mode = 'binary', # Real vs Fake
    shuffle = False
)

Fake image size: (32, 32)
Real image size: (32, 32)
Found 20000 images belonging to 2 classes.


In [ ]:
# Test model on test dataset
test_loss, test_accuracy = model.evaluate(test_generator)

print(f"Test Loss: {test_loss}")
print(f"Test Accuracy: {test_accuracy}")

/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


625/625 ━━━━━━━━━━━━━━━━━━━━ 16s 25ms/step - accuracy: 0.7710 - loss: 0.4706
Test Loss: 0.36346495151519775
Test Accuracy: 0.8373500108718872


BENCHMARK AGAINST VGG

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
batch_size = 32
train_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2) # Normalise pixel values, split 20% of data into validation set
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size = (96, 96),
    batch_size = batch_size,
    class_mode = 'binary', # Real vs Fake
    subset = 'training' # 80% of dataset
)

val_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size = (96, 96),
    batch_size = batch_size,
    class_mode = 'binary', # Real vs Fake
    subset = 'validation' # 80% of dataset
)

test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(96, 96),
    batch_size=32,
    class_mode='binary',
    shuffle=False  # Important if evaluating with metrics
)

Found 80000 images belonging to 2 classes.
Found 20000 images belonging to 2 classes.
Found 20000 images belonging to 2 classes.


In [ ]:
mobilenet_base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(96, 96, 3))
for layer in mobilenet_base.layers:
    layer.trainable = False

model = Sequential([
    mobilenet_base,
    GlobalAveragePooling2D(),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')  # Binary output
])

model.compile(optimizer=Adam(learning_rate=0.0001),
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.fit(train_generator, validation_data=val_generator, epochs=3)
model.evaluate(test_generator)

Epoch 1/3
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 0s 207ms/step - accuracy: 0.7918 - loss: 0.4341

/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


2500/2500 ━━━━━━━━━━━━━━━━━━━━ 653s 259ms/step - accuracy: 0.7918 - loss: 0.4341 - val_accuracy: 0.8881 - val_loss: 0.2712
Epoch 2/3
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 644s 258ms/step - accuracy: 0.8898 - loss: 0.2698 - val_accuracy: 0.9006 - val_loss: 0.2418
Epoch 3/3
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 657s 263ms/step - accuracy: 0.9004 - loss: 0.2442 - val_accuracy: 0.9064 - val_loss: 0.2262
625/625 ━━━━━━━━━━━━━━━━━━━━ 130s 208ms/step - accuracy: 0.9074 - loss: 0.2291


[0.22407183051109314, 0.9078999757766724]

In [ ]:
# Building GUI using Gradio
!pip install -q gradio

In [ ]:
import gradio as gr
from PIL import Image
import numpy as np

def image_predict(image):
    img = image.resize((32, 32))
    img = np.array(img) / 255.0
    img = np.expand_dims(img, axis=0)

    prediction = model.predict(img)
    label = (prediction > 0.5).astype("int32") # Converts (prediction > 0.5), which is a boolean value, into 1 or 0

    confidence = float(prediction[0][0])  # Extract the float from the array

    if label == 0:
        return f"""
          <div style="font-size: 18px;">
            <p><b>Prediction: <u>AI Generated</u></b></p>
            <b>Confidence:</b> {1-confidence:.2%}
          </div>
        """
    else:
        return f"""
          <div style="font-size: 18px;">
            <p><b>Prediction: <u>Not AI Generated</u></b></p>
            <b>Confidence:</b> {confidence:.2%}
          </div>
        """

interface = gr.Interface(
    fn=image_predict, # Choose model to run
    inputs=gr.Image(type="pil"), # Allows user to upload image
    outputs=gr.HTML(), # Outputs AI Generated or not
    title="50.021 Project: AI Generated or Not?",
    description="""
      <div style='text-align: center;'>Upload an image (.webp, .jpeg, .jpg, .png) to determine if it is Real, or AI Generated.<br><br>
      <div style='font-size: 10px; color: gray;'><i>Made by 1009942 Yong Jun Han</i></div>
      </div>
      """,
    allow_flagging="never"
)

interface.launch()

/usr/local/lib/python3.11/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated.Use `flagging_mode` instead.
  warnings.warn(


It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bd08f5ae7b4529b04e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
